In [7]:

%pip install tabulate


Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import idx2numpy
from pathlib import Path
from torch.utils.data import DataLoader, Dataset, random_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## 1. THIẾT LẬP THAM SỐ VÀ THIẾT BỊ

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 10
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
PATIENCE = 3

print(f"Đang sử dụng thiết bị: {device}")

Đang sử dụng thiết bị: cpu


## 2. XỬ LÝ DỮ LIỆU

In [3]:
DATA_DIR = Path("mnist-dataset")
TRAIN_IMAGES_PATH = DATA_DIR / "train-images-idx3-ubyte"
TRAIN_LABELS_PATH = DATA_DIR / "train-labels-idx1-ubyte"
TEST_IMAGES_PATH = DATA_DIR / "t10k-images-idx3-ubyte"
TEST_LABELS_PATH = DATA_DIR / "t10k-labels-idx1-ubyte"

class MnistIDXDataset(Dataset):
    def __init__(self, image_path: Path, label_path: Path) -> None:
        images = idx2numpy.convert_from_file(str(image_path))
        labels = idx2numpy.convert_from_file(str(label_path))
        
        # [N, 28, 28] -> float32 trong khoảng [0, 1]
        # Thêm kênh màu channel=1 cho CNN: [N, 1, 28, 28]
        self.images = torch.from_numpy(images.astype(np.float32) / 255.0).unsqueeze(1)
        self.labels = torch.from_numpy(labels.astype(np.int64))

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.images[index], self.labels[index]

# Tải dữ liệu
full_train_dataset = MnistIDXDataset(TRAIN_IMAGES_PATH, TRAIN_LABELS_PATH)
test_dataset = MnistIDXDataset(TEST_IMAGES_PATH, TEST_LABELS_PATH)

# Chia Train/Validation
VALIDATION_RATIO = 0.10
validation_size = int(len(full_train_dataset) * VALIDATION_RATIO)
train_size = len(full_train_dataset) - validation_size
train_dataset, validation_dataset = random_split(
    full_train_dataset,
    lengths=[train_size, validation_size],
    generator=torch.Generator().manual_seed(SEED),
)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 3. ĐỊNH NGHĨA CÁC MÔ HÌNH

In [4]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim=28*28, hidden_dim_1=256, hidden_dim_2=128, num_classes=10, dropout_rate=0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden_dim_1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim_2, num_classes)
        )

    def forward(self, x):
        return self.network(x)

class CNNClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Input: [batch_size, 1, 28, 28]
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: [batch_size, 32, 14, 14]
            
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: [batch_size, 64, 7, 7]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 4. HÀM HUẤN LUYỆN VÀ ĐÁNH GIÁ

In [5]:
def train_one_epoch(dataloader, model, loss_fn, optimizer, device):
    model.train()
    total_loss, total_samples = 0.0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size
    return total_loss / total_samples

def evaluate(dataloader, model, loss_fn, device):
    model.eval()
    total_loss, total_samples = 0.0, 0
    all_labels, all_preds = [], []
    with torch.inference_mode():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = loss_fn(logits, labels)
            preds = logits.argmax(dim=1)
            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    all_labels, all_preds = np.asarray(all_labels), np.asarray(all_preds)
    return {
        "loss": total_loss / total_samples,
        "f1": f1_score(all_labels, all_preds, average="macro", zero_division=0)
    }

## 5. CHƯƠNG TRÌNH CHÍNH (SO SÁNH)

In [6]:
results = []
models_to_test = {
    "MLP": MLPClassifier(),
    "CNN": CNNClassifier()
}

print("\nBắt đầu huấn luyện và so sánh...")
for model_name, model in models_to_test.items():
    print(f"\n--- Đang huấn luyện mô hình: {model_name} ---")
    model = model.to(device)
    num_params = count_trainable_parameters(model)
    print(f"Số lượng tham số: {num_params:,}")
    
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    best_f1 = -np.inf
    epochs_no_improve = 0
    start_time = time.time()
    
    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(train_loader, model, loss_fn, optimizer, device)
        val_res = evaluate(validation_loader, model, loss_fn, device)
        
        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_res['loss']:.4f} | Val F1: {val_res['f1']*100:.2f}%")
        
        if val_res["f1"] > best_f1:
            best_f1 = val_res["f1"]
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= PATIENCE:
            print(f"  -> Early stopping ở epoch {epoch}")
            break
            
    train_time = time.time() - start_time
    
    results.append({
        "Mô hình": model_name,
        "Số tham số": f"{num_params:,}",
        "Thời gian train (s)": f"{train_time:.2f}",
        "F1-score cao nhất (%)": f"{best_f1 * 100:.2f}%"
    })

print("\n==========================================")
print(" BẢNG SO SÁNH KẾT QUẢ GIỮA MLP VÀ CNN ")
print("==========================================")
df_results = pd.DataFrame(results)
print(df_results.to_markdown(index=False))


Bắt đầu huấn luyện và so sánh...

--- Đang huấn luyện mô hình: MLP ---
Số lượng tham số: 235,146
Epoch 01 | Train Loss: 0.3399 | Val Loss: 0.1580 | Val F1: 95.41%
Epoch 02 | Train Loss: 0.1423 | Val Loss: 0.1187 | Val F1: 96.20%
Epoch 03 | Train Loss: 0.1039 | Val Loss: 0.0948 | Val F1: 97.09%
Epoch 04 | Train Loss: 0.0822 | Val Loss: 0.0882 | Val F1: 97.35%
Epoch 05 | Train Loss: 0.0693 | Val Loss: 0.0794 | Val F1: 97.49%
Epoch 06 | Train Loss: 0.0594 | Val Loss: 0.0835 | Val F1: 97.48%
Epoch 07 | Train Loss: 0.0490 | Val Loss: 0.0745 | Val F1: 97.85%
Epoch 08 | Train Loss: 0.0473 | Val Loss: 0.0790 | Val F1: 97.73%
Epoch 09 | Train Loss: 0.0426 | Val Loss: 0.0757 | Val F1: 97.90%
Epoch 10 | Train Loss: 0.0368 | Val Loss: 0.0735 | Val F1: 97.94%

--- Đang huấn luyện mô hình: CNN ---
Số lượng tham số: 421,642
Epoch 01 | Train Loss: 0.2442 | Val Loss: 0.0631 | Val F1: 98.09%
Epoch 02 | Train Loss: 0.0741 | Val Loss: 0.0607 | Val F1: 98.16%
Epoch 03 | Train Loss: 0.0546 | Val Loss: 0.04

ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.